# Milestone 4 lab, notebook 2 of 8: How serious is it

## Where we are

| Notebook | What you do |
|---|---|
| 1. A rule lives in dbt | Move a check out of a notebook and into a file that dbt runs |
| **2. How serious is it** (you are here) | Tell dbt which failures stop the night. Keep the rows a test returns |
| 3. Test the SQL, not the data | Check your own query on four rows typed by hand |
| 4. A gate, and where it stands | Run one rule in three places, and see what each place can see |
| 5. Warn, block or set aside | Decide what a failure does. Then block a real run in Airflow |
| 6. Quarantine | Set the bad rows aside with a reason, and let the good rows go on |
| 7. Test the tests | Plant a fault, and prove its gate catches it. One check in four tools |
| 8. The production version | Run the whole week through Airflow. See what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import (
    after_the_gates, all_tests, before_the_clean_gate, dbt, prepare_week, show_file, sql,
)

prepare_week()
before_the_clean_gate()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August, through three gates.
Sunday night ran 47 tests. 1 of 6 stores can be paid.
The clean tables are rebuilt from the raw rows. No gate has looked at them yet.


## Step 1: Two failures that are not the same size

> **Who does what.** dbt holds the rule: a test is a query that returns the rows that break the rule. Airflow runs the rule and acts on the result: when the check runs, what waits for it, and what happens when it fails.

Two things went wrong at Lotus on Friday 28 August.

- Store `S04` sold twelve packs of butter at yesterday's price. The store is **$6.00** short.
- Suppose the stores table had lost a row that night. Every sale of one store would vanish from every report.

Both are failed checks. Should both stop the pipeline at 02:00? The first costs six dollars, and the published numbers are right. The second makes every number wrong.

A test must be able to say how serious it is.

## Step 2: Severity

Look at the first lines of two tests.

In [2]:
for name in ("price_of_the_day", "six_stores_once"):
    show_file(f"dbt_project/tests/clean/{name}.sql", "{{ config", 1)
    print()

# dbt_project/tests/clean/price_of_the_day.sql, from line 6
{{ config(tags=['clean'], severity='warn', meta={'number': 8, 'action': 'warn', 'lens': 'business', 'question': 'Accurate?', 'protects': 'Finance and the store: money the store should have taken and did not'}) }}

# dbt_project/tests/clean/six_stores_once.sql, from line 6
{{ config(tags=['clean'], severity='error', meta={'number': 22, 'action': 'block', 'lens': 'data', 'question': 'Unique?', 'protects': 'Everyone who reads the two outputs: a sales line is joined to this table. A store that is there twice doubles its sales'}) }}



Find the word `severity` in each.

- `severity='warn'`: when the test finds rows, dbt reports it and **goes on**.
- `severity='error'`: when the test finds rows, dbt reports an error. The gate that started dbt **stops**.

That is the test's **severity**: how serious a failure is for dbt.

See both. First the butter.

In [3]:
dbt("test", "--select", "price_of_the_day")

Found 4 models, 46 data tests, 25 sources, 480 macros, 2 unit tests
1 of 1 WARN 6 price_of_the_day  ...  [WARN 6 in 0.04s]
Done. PASS=0 WARN=1 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1


`WARN 6`: six lines of butter. The last line says `WARN=1 ERROR=0`. For dbt, the run went fine.

Now the other kind. The standard check "the key is unique" has `severity: error`. Give it something to find: copy one sales line in the clean table, as you did in milestone 1.

In [4]:
from lab_helpers import change

change("INSERT INTO clean.sales_lines SELECT * FROM clean.sales_lines WHERE line_id = 'S03-0824-000001'")
dbt("test", "--select", "standard_key_is_unique")

Found 4 models, 46 data tests, 25 sources, 480 macros, 2 unit tests
1 of 1 FAIL 1 standard_key_is_unique  ...  [FAIL 1 in 0.04s]
Done. PASS=0 WARN=0 ERROR=1 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1


`FAIL 1`, and `ERROR=1`. Same kind of query, same kind of result: rows came back. The only difference is the word in the config.

Here is how the 42 test files of the project are set.

In [5]:
tests = all_tests()
tests.groupby(["severity", "action"]).size().rename("tests").reset_index()

,severity,action,tests
0,error,block,11
1,warn,quarantine,17
2,warn,refuse_file,1
3,warn,warn,13


Only `block` is an `error` for dbt. Everything else is a `warn`: dbt goes on, and the **pipeline** reads the `action` and does the rest. `quarantine`, `refuse_file` and `warn` are not dbt's words. They are this project's, and notebooks 5 and 6 are about them.

## Step 3: Which rows? Stored failures

"WARN 6" tells you how many. It does not tell you which. At 02:00 a count is not enough: somebody has to look at the rows, and the pipeline has to act on those exact rows.

This project asks dbt to keep them. One line in `dbt_project.yml`:

In [6]:
show_file("dbt_project/dbt_project.yml", "data_tests:", 4)

# dbt_project/dbt_project.yml, from line 26
data_tests:
  lotus:
    +store_failures: true      # keep the rows a test returns, in a table named after the test
    +schema: audit             # ...in the schema `audit`


With that line, dbt saves the rows each test returns in a table named after the test, in the schema `audit`. Run the clean tests again, then look.

In [7]:
dbt("test", "--select", "tag:clean", only_failed=True)
sql("SELECT * FROM audit.refund_gives_money_back")

Found 4 models, 46 data tests, 25 sources, 480 macros, 2 unit tests
1 of 39 WARN 1 a_count_for_every_day  ...  [WARN 1 in 0.24s]
2 of 39 WARN 1 believable_quantity  ...  [WARN 1 in 0.02s]
8 of 39 WARN 3 every_sale_has_a_product  ...  [WARN 3 in 0.03s]
11 of 39 WARN 4 inside_opening_hours  ...  [WARN 4 in 0.03s]
12 of 39 WARN 1 line_adds_up  ...  [WARN 1 in 0.02s]
16 of 39 WARN 1 normal_volume  ...  [WARN 1 in 0.02s]
18 of 39 WARN 1 one_person_once  ...  [WARN 1 in 0.02s]
22 of 39 WARN 6 price_of_the_day  ...  [WARN 6 in 0.02s]
24 of 39 WARN 1 refund_gives_money_back  ...  [WARN 1 in 0.02s]
31 of 39 FAIL 1 standard_key_is_unique  ...  [FAIL 1 in 0.02s]
39 of 39 WARN 172 true_key_unique  ...  [WARN 172 in 0.05s]
(28 tests passed, and are not listed)
Done. PASS=28 WARN=10 ERROR=1 SKIP=0 NO-OP=0 REUSED=0 TOTAL=39


,line_id,store_id,sales_date,line_type,quantity,amount_cents
0,S03-0824-000008,S03,2026-08-24,refund,-1,949


First the list: the same ten tests as in notebook 1, and one more. `FAIL 1 standard_key_is_unique` is the line you copied a minute ago.

Then the table: there is the refund. The test's rows are a table now. So are the six lines of butter:

In [8]:
sql("SELECT line_id, store_id, sales_date, product_name, charged_cents, price_list_cents, store_short_cents FROM audit.price_of_the_day ORDER BY line_id")

,line_id,store_id,sales_date,product_name,charged_cents,price_list_cents,store_short_cents
0,S04-0828-000010,S04,2026-08-28,Butter 250g,349,399,100
1,S04-0828-000025,S04,2026-08-28,Butter 250g,349,399,100
2,S04-0828-000042,S04,2026-08-28,Butter 250g,349,399,100
3,S04-0828-000066,S04,2026-08-28,Butter 250g,349,399,100
4,S04-0828-000079,S04,2026-08-28,Butter 250g,349,399,100
5,S04-0828-000110,S04,2026-08-28,Butter 250g,349,399,100


These kept rows are called **stored failures**.

| | Without stored failures | With stored failures |
|---|---|---|
| What you know after the run | "1 test failed with 6 rows" | The 6 rows |
| To see the rows | Find the test's SQL and run it again | `SELECT * FROM audit.price_of_the_day` |
| What a program can do next | Stop, or not stop | Take exactly those rows out |

The last line is the important one. Look at how the pipeline uses the table. This is from `lotus_pipeline/gates.py`:

In [9]:
show_file("lotus_pipeline/gates.py", "def _set_sales_lines_aside", 8)

# lotus_pipeline/gates.py, from line 138
def _set_sales_lines_aside(wh: Warehouse, tests: list[TestResult], day: date) -> None:
    for test in tests:
        wh.execute(
            f"INSERT INTO quarantine.sales_lines SELECT l.*, ?, ?, ?, CAST(? AS DATE) "
            f"FROM clean.sales_lines l WHERE l.line_id IN (SELECT line_id FROM {test.table})",
            (test.name, test.title, test.is_a_copy, day),
        )
    # Too much of one file is bad: do not trust the rest of it either.


`{test.table}` is `audit.` and the name of the test. The pipeline takes the lines that dbt stored, and moves them aside. dbt found them. The pipeline acts.

One warning about these tables: dbt writes them again on every run. They hold the last run only. They are not a history.

Put the warehouse back.

In [10]:
after_the_gates()

Sunday's night ran again. The warehouse is as the nightly run left it.


## What you now know

- **Severity** is how serious a failed test is for dbt: `error` stops the gate, `warn` lets the run go on.
- A wrong price at one store and a broken stores table are both failures. They are not the same size.
- **Stored failures** are the rows a test returned, kept by dbt in a table named after the test.
- A count tells you something is wrong. The rows let a person look, and let the pipeline act.

**Say it in an interview:** "Each dbt test had a severity. A broken reference table was an error and stopped the run. A store charging yesterday's price was a warning, because the published numbers were still right. We stored the failing rows, so the pipeline could set exactly those rows aside."

**The theory:** [dbt Tests: Generic, Singular & Custom](https://www.d8loop.com/modules/knowledge-corner/dbt/dbt-tests-generic-singular-and-custom), and [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "Warn or block".

Next: [notebook 3](03_test_the_sql_not_the_data.ipynb). Your test's SQL is code. Who tests the test?